In [3]:
import numpy as np
import pandas as pd



In [4]:
df = pd.read_csv("E-Commerce raw.csv")


In [5]:
df.head()

,Order_ID,Order_Date,Customer_ID,Product,Category,Quantity,Unit_Price,Discount,Revenue,Cost,Profit,City,Customer_Segment,Payment_Method
0,ORD09130,2024-03-23,CUST02722,Blender,Home Appliances,8,67.17,0.05,510.49,414.00,96.49,Chennai,Home Office,Credit Card
1,ORD04526,2023-06-03,CUST00659,Jacket,Fashion,1,152.46,0.25,114.34,91.14,23.20,Chennai,Corporate,Cash
2,ORD01745,2023-06-18,CUST00162,Headphones,Electronics,6,183.0,0.20,878.40,980.40,-102.00,Bhubaneswar,Consumer,Debit Card
3,ORD06248,2025-06-16,CUST01141,Smartphone,Electronics,4,1202.5,0.00,4810.00,3392.36,1417.64,Bangalore,Corporate,Debit Card
4,ORD07863,2023-02-11,CUST03469,Shoes,Fashion,1,134.93,0.00,134.93,85.82,49.11,Bhubaneswar,Consumer,Credit Card


In [7]:
df.shape

(12150, 14)

In [8]:
df.dtypes

Order_ID             object
Order_Date           object
Customer_ID          object
Product              object
Category             object
Quantity              int64
Unit_Price           object
Discount            float64
Revenue             float64
Cost                float64
Profit              float64
City                 object
Customer_Segment     object
Payment_Method       object
dtype: object

In [9]:
df.isnull().sum()

Order_ID              0
Order_Date            0
Customer_ID         330
Product               0
Category              0
Quantity              0
Unit_Price           97
Discount            244
Revenue               0
Cost                  0
Profit                0
City                146
Customer_Segment      0
Payment_Method      122
dtype: int64

In [10]:
(df.isnull().sum() / len(df) * 100).round(2)

Order_ID            0.00
Order_Date          0.00
Customer_ID         2.72
Product             0.00
Category            0.00
Quantity            0.00
Unit_Price          0.80
Discount            2.01
Revenue             0.00
Cost                0.00
Profit              0.00
City                1.20
Customer_Segment    0.00
Payment_Method      1.00
dtype: float64

In [11]:
df.duplicated().sum()

np.int64(150)

In [12]:
df = df.drop_duplicates()

In [13]:
df.duplicated().sum()

np.int64(0)

In [14]:
text_columns = [
    "Order_ID",
    "Customer_ID",
    "Product",
    "Category",
    "City",
    "Customer_Segment",
    "Payment_Method"
]

for col in text_columns:
    df[col] = df[col].astype("string").str.strip()

In [15]:
df["City"] = df["City"].str.strip().str.title()

df["Category"] = df["Category"].str.strip().str.title()

df["Payment_Method"] = df["Payment_Method"].str.strip()

In [16]:
df["Payment_Method"] = df["Payment_Method"].replace({
    "upi": "UPI",
    "Upi": "UPI",
    "CreditCard": "Credit Card",
    "credit card": "Credit Card",
    "CREDIT CARD": "Credit Card"
})

In [17]:
df["Order_Date"] = pd.to_datetime(
    df["Order_Date"],
    errors="coerce",
    format="mixed"
)

In [18]:
df["Order_Date"].dtype

dtype('<M8[ns]')

In [19]:
df["Order_Date"].isnull().sum()

np.int64(0)

In [20]:
df["Unit_Price"] = (
    df["Unit_Price"]
    .astype("string")
    .str.replace("$", "", regex=False)
    .str.replace(",", "", regex=False)
    .str.strip()
)

In [21]:
df["Unit_Price"] = pd.to_numeric(
    df["Unit_Price"],
    errors="coerce"
)

In [22]:
df["Unit_Price"].dtype

Float64Dtype()

In [23]:
numeric_columns = [
    "Quantity",
    "Unit_Price",
    "Discount",
    "Revenue",
    "Cost",
    "Profit"
]

for col in numeric_columns:
    df[col] = pd.to_numeric(
        df[col],
        errors="coerce"
    )

In [24]:
df[numeric_columns].dtypes

Quantity        int64
Unit_Price    Float64
Discount      float64
Revenue       float64
Cost          float64
Profit        float64
dtype: object

In [25]:
df["Customer_ID"] = df["Customer_ID"].fillna("Unknown")

In [26]:
df["City"] = df["City"].fillna(
    df["City"].mode()[0]
)

In [27]:
df["Payment_Method"] = df["Payment_Method"].fillna(
    df["Payment_Method"].mode()[0]
)

In [28]:
df["Unit_Price"] = df.groupby("Product")["Unit_Price"].transform(
    lambda x: x.fillna(x.median())
)

In [29]:
df["Unit_Price"] = df["Unit_Price"].fillna(
    df["Unit_Price"].median()
)

In [30]:
df["Discount"] = df.groupby("Category")["Discount"].transform(
    lambda x: x.fillna(x.median())
)

In [31]:
df["Discount"] = df["Discount"].fillna(
    df["Discount"].median()
)

In [32]:
df.isnull().sum()

Order_ID            0
Order_Date          0
Customer_ID         0
Product             0
Category            0
Quantity            0
Unit_Price          0
Discount            0
Revenue             0
Cost                0
Profit              0
City                0
Customer_Segment    0
Payment_Method      0
dtype: int64

In [34]:
df[df["Quantity"] <= 0]

,Order_ID,Order_Date,Customer_ID,Product,Category,Quantity,Unit_Price,Discount,Revenue,Cost,Profit,City,Customer_Segment,Payment_Method


In [35]:
df.loc[df["Quantity"] <= 0, "Quantity"] = np.nan

In [36]:
df["Quantity"] = df["Quantity"].fillna(
    df["Quantity"].median()
)

In [37]:
df[
    (df["Discount"] < 0) |
    (df["Discount"] > 1)
]

,Order_ID,Order_Date,Customer_ID,Product,Category,Quantity,Unit_Price,Discount,Revenue,Cost,Profit,City,Customer_Segment,Payment_Method


In [38]:
df.loc[
    (df["Discount"] < 0) |
    (df["Discount"] > 1),
    "Discount"
] = np.nan

In [39]:
df["Discount"] = df["Discount"].fillna(
    df["Discount"].median()
)

In [40]:
def cap_outliers_iqr(df, column):
    
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)

    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    df[column] = df[column].clip(
        lower=lower,
        upper=upper
    )

    print(column)
    print("Lower limit:", lower)
    print("Upper limit:", upper)

In [41]:
outlier_columns = [
    "Quantity",
    "Unit_Price",
    "Revenue",
    "Cost"
]

for col in outlier_columns:
    cap_outliers_iqr(df, col)

Quantity
Lower limit: -5.5
Upper limit: 14.5
Unit_Price
Lower limit: -317.7087500000001
Upper limit: 810.8012500000001
Revenue
Lower limit: -1495.9125000000001
Upper limit: 3315.4275
Cost
Lower limit: -1215.98
Upper limit: 2710.1800000000003


In [42]:
df["Profit"] = df["Revenue"] - df["Cost"]

In [43]:
df["Order_Year"] = df["Order_Date"].dt.year

In [44]:
df["Order_Month"] = df["Order_Date"].dt.month

In [45]:
df["Order_Month_Name"] = df["Order_Date"].dt.month_name()

In [46]:
df["Profit_Margin"] = np.where(
    df["Revenue"] != 0,
    (df["Profit"] / df["Revenue"]) * 100,
    0
)

In [47]:
df["Profit_Margin"] = df["Profit_Margin"].round(2)

In [48]:
financial_columns = [
    "Unit_Price",
    "Discount",
    "Revenue",
    "Cost",
    "Profit",
    "Profit_Margin"
]

df[financial_columns] = df[financial_columns].round(2)

In [49]:
df.shape

(12000, 18)

In [50]:
df.isnull().sum()

Order_ID            0
Order_Date          0
Customer_ID         0
Product             0
Category            0
Quantity            0
Unit_Price          0
Discount            0
Revenue             0
Cost                0
Profit              0
City                0
Customer_Segment    0
Payment_Method      0
Order_Year          0
Order_Month         0
Order_Month_Name    0
Profit_Margin       0
dtype: int64

In [51]:
df.columns

Index(['Order_ID', 'Order_Date', 'Customer_ID', 'Product', 'Category',
       'Quantity', 'Unit_Price', 'Discount', 'Revenue', 'Cost', 'Profit',
       'City', 'Customer_Segment', 'Payment_Method', 'Order_Year',
       'Order_Month', 'Order_Month_Name', 'Profit_Margin'],
      dtype='object')

In [52]:
df.to_csv(
    "clean_dataset.csv",
    index=False
)

In [54]:
clean_df = pd.read_csv("clean_dataset.csv")

In [55]:
clean_df.head()


,Order_ID,Order_Date,Customer_ID,Product,Category,Quantity,Unit_Price,Discount,Revenue,Cost,Profit,City,Customer_Segment,Payment_Method,Order_Year,Order_Month,Order_Month_Name,Profit_Margin
0,ORD09130,2024-03-23,CUST02722,Blender,Home Appliances,8.0,67.17,0.05,510.49,414.00,96.49,Chennai,Home Office,Credit Card,2024,3,March,18.90
1,ORD04526,2023-06-03,CUST00659,Jacket,Fashion,1.0,152.46,0.25,114.34,91.14,23.20,Chennai,Corporate,Cash,2023,6,June,20.29
2,ORD01745,2023-06-18,CUST00162,Headphones,Electronics,6.0,183.00,0.20,878.40,980.40,-102.00,Bhubaneswar,Consumer,Debit Card,2023,6,June,-11.61
3,ORD06248,2025-06-16,CUST01141,Smartphone,Electronics,4.0,810.80,0.00,3315.43,2710.18,605.25,Bangalore,Corporate,Debit Card,2025,6,June,18.26
4,ORD07863,2023-02-11,CUST03469,Shoes,Fashion,1.0,134.93,0.00,134.93,85.82,49.11,Bhubaneswar,Consumer,Credit Card,2023,2,February,36.40


In [57]:
clean_df.isnull().sum()

Order_ID            0
Order_Date          0
Customer_ID         0
Product             0
Category            0
Quantity            0
Unit_Price          0
Discount            0
Revenue             0
Cost                0
Profit              0
City                0
Customer_Segment    0
Payment_Method      0
Order_Year          0
Order_Month         0
Order_Month_Name    0
Profit_Margin       0
dtype: int64